In [ ]:
import os
import sys
import traceback
import zipfile


print("=" * 60)
print("NEMOTRON LORA v50 — KAGGLE COMPATIBLE")
print("No pip installs. Uses: torch, transformers, peft, datasets")
print("=" * 60)

try:
    import kagglehub
    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    print("[1/6] All standard dependencies loaded")

    # Hardware
    print("\n[2/6] Hardware:")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {p.name} ({p.total_memory / 1024**3:.1f} GB)")
    else:
        print("  CPU only")

    # Load data (ROBUST)
    print("\n[3/6] Loading training data...")
    train_file = None
    for base in ["/kaggle/input/nvidia-nemotron-model-reasoning-challenge", "/kaggle/input"]:
        if os.path.exists(base):
            for root, dirs, files in os.walk(base):
                for f in files:
                    if f.lower() == "train.csv":
                        train_file = os.path.join(root, f)
                        break
                if train_file:
                    break
        if train_file:
            break

    if not train_file:
        print("ERROR: train.csv not found")
        sys.exit(1)

    df = pd.read_csv(train_file)
    print(f"  {len(df)} rows, cols: {list(df.columns)}")

    # Format for causal LM training
    print("\n[4/6] Formatting training data...")
    texts = []
    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"]).strip()
        # The eval metric extracts \boxed{answer}, so train with that format
        text = f"Solve this reasoning problem.\n\n{prompt}\n\nTherefore, the answer is \\boxed{{{answer}}}."
        texts.append(text)

    print(f"  {len(texts)} training texts prepared")

    # Tokenize
    print("\n[5/6] Loading model and tokenizing...")

    # Kaggle downloads via model_download when model is pre-attached
    model_id = "metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1"
    print(f"  Downloading {model_id}...")
    model_path = kagglehub.model_download(model_id)
    print(f"  Model path: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    # LoRA config
    lora = LoraConfig(
        r=32,
        lora_alpha=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    # Dataset
    dataset = Dataset.from_dict({"text": texts})

    def tok(examples):
        return tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")

    tokenized = dataset.map(tok, batched=True, remove_columns=["text"])
    # Add labels for causal LM (labels = input_ids)
    tokenized = tokenized.map(lambda x: {"labels": x["input_ids"]}, batched=True)
    print(f"  Dataset ready: {len(tokenized)} examples")

    # Train
    print("\n[6/6] Training...")
    args = TrainingArguments(
        output_dir="./nemotron_lora_adapter",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
    trainer.train()

    # Save
    print("\nSaving adapter...")
    adapter_dir = "./nemotron_lora_adapter"
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    # Verify adapter_config.json
    cfg = os.path.join(adapter_dir, "adapter_config.json")
    if not os.path.exists(cfg):
        raise FileNotFoundError("adapter_config.json missing!")
    print(f"  adapter_config.json: {os.path.getsize(cfg)} bytes")

    # Create submission.zip
    print("\nCreating submission.zip...")
    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(adapter_dir):
            for f in files:
                fp = os.path.join(root, f)
                ar = os.path.relpath(fp, adapter_dir)
                zf.write(fp, ar)
                print(f"  + {ar}")

    sz = os.path.getsize("submission.zip")
    print(f"\n{'=' * 60}")
    print(f"SUBMISSION READY: submission.zip ({sz / 1024:.1f} KB)")
    print(f"{'=' * 60}")
    # Final verify
    with zipfile.ZipFile("submission.zip", "r") as zf:
        names = zf.namelist()
    assert any("adapter_config" in n for n in names), "No adapter_config.json!"
    print("✓ Verified: contains adapter files")

except Exception as e:
    print(f"\nERROR: {e}")
    traceback.print_exc()
    sys.exit(1)